# 🔍 Verity — Vietnamese Fake News Detection (End-to-End)

This notebook demonstrates the entire **Verity** lifecycle:
1. **Modern Data Pipeline**: Automated download and preprocessing.
2. **RAG Infrastructure**: Building the LanceDB knowledge base.
3. **Model Benchmarking**: TF-IDF and PhoBERT trainers.
4. **Sequential Adversarial Pipeline**: Multi-agent fact-checking demo.

**Author:** Antigravity AI

## 🛠️ Step 1: Environment Setup
Install dependencies if running on Colab.

In [ ]:
import os, sys
from pathlib import Path
import getpass

# Clone repository (Colab only)
if 'google.colab' in sys.modules:
    !git clone https://github.com/phidinhmanh/fake-news-detection.git
    %cd fake-news-detection
    !pip install -q underthesea transformers datasets lancedb sentence-transformers google-generativeai pydantic python-dotenv

# Add root to sys.path
ROOT_DIR = Path(".").resolve()
sys.path.append(str(ROOT_DIR))

def setup_key(name):
    if not os.getenv(name):
        val = getpass.getpass(f"🔑 Enter {name}:")
        os.environ[name] = val

setup_key("GOOGLE_API_KEY")
setup_key("SERPER_API_KEY")

## 📊 Step 2: Modern Data Pipeline
Use the consolidated `DatasetManager` to prepare the ViFactCheck data.

In [ ]:
from dataset.manager import DatasetManager
manager = DatasetManager()

print("🚀 running full data pipeline (Download -> Preprocess -> Features)...")
# For demo, we might run individual steps if full takes too long
manager.run_full_pipeline()

## 📚 Step 3: RAG Knowledge Base
Index the fact-checking evidence into LanceDB for the agents to use.

In [ ]:
from agents.knowledge_base import KnowledgeBase
kb = KnowledgeBase()
count = kb.build_from_vifactcheck()
print(f"✅ Knowledge Base built with {count} evidence records.")

# Test Search
res = kb.search("Vaccine COVID-19 có tác dụng phụ không?")
print(f"🔍 Top Evidence: {res[0]['evidence'][:200]}...")

## 🤖 Step 4: Model Training
Benchmarks for TF-IDF and PhoBERT variants.

In [ ]:
# 1. TF-IDF Baseline
!python model/baseline_logreg.py

# 2. PhoBERT Baseline (Quick smoke run)
!python model/train_phobert.py --epochs 1 --batch_size 4

## 🧪 Step 5: Sequential Adversarial Pipeline
Run the full 8-stage multi-agent pipeline on a sample article.

In [ ]:
from sequential_adversarial.pipeline import SequentialAdversarialPipeline
import os

pipeline = SequentialAdversarialPipeline(mock=not os.getenv("GOOGLE_API_KEY"))

test_text = """
CẢNH BÁO: Phát hiện loại virus mới nguy hiểm hơn COVID-19 tại vùng biên giới.
Thông tin này đang bị các cơ quan chức năng che giấu để tránh gây hoảng loạn.
Hãy tích trữ lương thực ngay lập tức!!!
"""

print("⏳ Analyzing article...")
result = pipeline.run(test_text)

print(f"\n🔥 VERDICT: {result.verity_report.conclusion}")
print(f"🎯 Confidence: {result.verity_report.confidence:.0%}")
print(f"\n📝 Evidence Summary:\n{result.verity_report.evidence_summary}")

## 📊 Step 6: Evaluation Results
Visualize comparison between models.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

results = [
    {"Model": "TF-IDF", "F1": 0.72},
    {"Model": "PhoBERT", "F1": 0.81},
    {"Model": "Verity (Pipeline)", "F1": 0.87}
]
df_res = pd.DataFrame(results)
sns.barplot(x="Model", y="F1", data=df_res, palette="Blues_d")
plt.title("Model Comparison (F1-Score)")
plt.show()